# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [ ]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [ ]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-4o-mini'
openai = OpenAI()

In [ ]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [ ]:
ed = Website("https://edwarddonner.com")
print(ed.get_contents())
ed.links

## First step: Have GPT-4o-mini figure out which links are relevant

### Use a call to gpt-4o-mini to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [ ]:
# This is called "One shot prompting"

link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}
"""

In [ ]:
print(link_system_prompt)

In [ ]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links, javascript code.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [ ]:
print(get_links_user_prompt(ed))

In [ ]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [ ]:
get_links("https://tourno.fi")

In [ ]:
# Anthropic has made their site harder to scrape, so I'm using HuggingFace..

huggingface = Website("https://huggingface.co")
huggingface.links

In [ ]:
get_links("https://huggingface.co")

## Second step: make the brochure!

Assemble all the details into another prompt to GPT4-o

In [87]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [88]:
print(get_all_details("https://tourno.fi"))

Found links: {'links': [{'type': 'about page', 'url': 'https://tourno.fi/about_tournoco.php'}]}
Landing page:
Webpage Title:
tourno | Home
Webpage Contents:
SIGN IN
REGISTER
English
Suomi
About Us
Pricing
New Tournaments
All Tournaments
Home
About Us
Pricing
New Tournaments
All Tournaments
27
TOURNAMENTS
954
TEAMS
2064
GAMES
8265
GOALS
APU
BK-46
Borgå Akilles
EBK
EIF
EPS
EsPa
Espoon Tikka
FC Futura
FC Haka
FC Halikko
FC HIK
FC Honka
FC Kasiysi
FC Kirkkonummi
FC Kontu
FC Kuusysi
FC Legirus Inter
FC Lohja
FC LU
FC Nokia
FC POHU
FC Rauma
FC Reipas
FC SauPa
FC Viikingit
FC WILD
Gnistan
GrIFK
HIFK
HJK
HJS
HooGee
HPS
HyPS
IIF
Ilves
JäPS
Kaarinan Pojat
KaPy
KeiKa
KelA
KJS
KoiPS
Kopse
KP-75
KP-75/TUPS YJ
KSF
Kuusysi
KyIF
KäPä
Lahen Pojat
LeKi-Futis
LePa
LoPa
LPS
Maski
MPS
NJS
Nõmme Kalju
NoPS
NuPS
Orimattilan Pedot
Pallo-Iirot
PEP
PK-35
PKKU
Porvoon Akilles
PPJ
PPS
PPV
PuiU
RaiFu
RaKe
RiPs
Salpa
Sapa
Sibbo Vargarna
Siuntion Sisu
TiPS
ToBK
TP-49
TPS
TuNL
TuPs
Valtti
ViTa
VJS
Väst4
Wilpas
ÅIFK
R

In [105]:
system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
# and creates a short humorous, entertaining, jokey brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
# Include details of company culture, customers and careers/jobs if you have the information."


In [106]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [107]:
get_brochure_user_prompt("Tourno.fi", "https://tourno.fi")

Found links: {'links': [{'type': 'about page', 'url': 'https://tourno.fi/about_tournoco.php'}]}


'You are looking at a company called: Tourno.fi\nHere are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\nLanding page:\nWebpage Title:\ntourno | Home\nWebpage Contents:\nSIGN IN\nREGISTER\nEnglish\nSuomi\nAbout Us\nPricing\nNew Tournaments\nAll Tournaments\nHome\nAbout Us\nPricing\nNew Tournaments\nAll Tournaments\n27\nTOURNAMENTS\n954\nTEAMS\n2064\nGAMES\n8265\nGOALS\nAPU\nBK-46\nBorgå Akilles\nEBK\nEIF\nEPS\nEsPa\nEspoon Tikka\nFC Futura\nFC Haka\nFC Halikko\nFC HIK\nFC Honka\nFC Kasiysi\nFC Kirkkonummi\nFC Kontu\nFC Kuusysi\nFC Legirus Inter\nFC Lohja\nFC LU\nFC Nokia\nFC POHU\nFC Rauma\nFC Reipas\nFC SauPa\nFC Viikingit\nFC WILD\nGnistan\nGrIFK\nHIFK\nHJK\nHJS\nHooGee\nHPS\nHyPS\nIIF\nIlves\nJäPS\nKaarinan Pojat\nKaPy\nKeiKa\nKelA\nKJS\nKoiPS\nKopse\nKP-75\nKP-75/TUPS YJ\nKSF\nKuusysi\nKyIF\nKäPä\nLahen Pojat\nLeKi-Futis\nLePa\nLoPa\nLPS\nMaski\nMPS\nNJS\nNõmme Kalju\nNoPS\nNuPS\nOrimattilan Ped

In [99]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [100]:
create_brochure("Tourno.fi", "https://tourno.fi")

Found links: {'links': [{'type': 'about page', 'url': 'https://tourno.fi/about_tournoco.php'}]}


```markdown
# Welcome to Tourno.fi: Where Tournaments Are a Kick!

## ⚽ Join the Fun!
Welcome to Tourno.fi, the website where goals are scored and dreams are made… or at least where people spend quality time trying! Whether you're a soccer fanatic or just someone who accidentally signed up while looking for a cat video, we guarantee you’ll have a blast.

## 🎉 What’s Cooking?
With **27 tournaments**, **954 teams**, **2064 games**, and a whopping **8265 goals** scored, it's safe to say we don't play around. Wait… actually, we do! Just look at those numbers! 

**Coming Up:**
- 🗓️ **KodinTavaratalo Cup 2025** - Be there or be square! (Only kidding, square people are welcome too).  
- 📍 **Location:** Karjaa, because what sounds cooler than saying "I went to a tournament in Karjaa"?

## 🏆 Our Teams Are All Over!
We're a big happy (and sometimes chaotic) family! Our teams include the spirited warriors from **FC HJK**, the starry-eyed adventurers at **FC Honka**, and many more. They say teamwork makes the dream work, and with this many teams, we’ve got enough dreams to fill a stadium! 

### Some Customer Love:
> "I only signed up to watch the food stalls — ended up with some new best friends (and a little bit of a sunburn)." - A Satisfied Customer

## 💼 Careers & Company Culture
**Looking to join the madness?** We’re on the lookout for folks who can kick it with us! 
- **Goal-Oriented Cats:** Must love soccer and proper use of “your” and “you’re.”
- **Eventulated Wizards:** If you think you can spin plates and juggle schedules, you belong here!

### Culture:
At Tourno.fi, we embrace the beautiful chaos that soccer brings. Every day is a game day, and we place bets on who can make the best cup of office coffee (pro tip: it’s usually the one with the most caffeine). We work hard but play harder. Bring your soccer socks, and let’s have some fun!

## 🎈 Why Tourno.fi?
- **Tournaments:** We have tournaments coming out of our ears — not literally, that sounds painful.
- **Friendly Faces:** No grumpy goalkeepers here! Just smiles and the occasional ref whistle.
- **Community Spirit:** Join a group of like-minded soccer aficionados who believe in cooperating towards one ultimate goal – having an incredible time!

## 📞 Contact Us!
Ready to kick off your journey with us? Sign in or register at [Tourno.fi](https://tourno.fi). We promise there’s a tournament waiting just for you, and no red cards involved! 🎉

### *Join the fun, kick the ball, and let’s score some memories together!*
```


## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [108]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

In [109]:
stream_brochure("Craig Rothwell", "https://www.linkedin.com/in/rothwellcraig/")

Found links: {'links': []}


# Craig Rothwell Company Brochure

## About Us
Welcome to **Craig Rothwell**, where innovation meets excellence. While our website currently lacks a title, our mission and dedication to providing quality services and products shine through in everything we do. We take pride in delivering unparalleled solutions tailored to meet the unique needs of our diverse clientele.

## Company Culture
At Craig Rothwell, we foster a collaborative and inclusive workplace environment where creativity and teamwork thrive. Our employees are our greatest asset, and we believe in empowering them to excel. Our company culture emphasizes continuous learning, respect, and integrity. We encourage open communication and actively seek feedback to enhance our work environment and processes.

## Our Customers
We serve a wide array of customers from various industries, including technology, healthcare, and retail. Our commitment to customer satisfaction ensures that we listen to our clients’ needs and provide tailored solutions that drive results. Whether you are a small business looking for innovative solutions or a large corporation seeking efficiency, Craig Rothwell is here to support you.

## Careers at Craig Rothwell
We are always on the lookout for talented and driven individuals to join our growing team. At Craig Rothwell, we offer exciting career opportunities across various fields. By joining our team, you can expect:

- A supportive and dynamic work environment
- Opportunities for professional growth and development
- Competitive compensation and benefits
- A commitment to work-life balance

If you are ready to take the next step in your career and be part of a thriving company, we invite you to explore our job openings and apply today!

## Get in Touch
For more information about our services, company culture, or career opportunities, please don’t hesitate to reach out. We look forward to collaborating with you and achieving success together!

---

*Note: This brochure can be expanded with specific service offerings, testimonials, or case studies to enhance its appeal.*

In [103]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("Tourno.fi", "https://tourno.fi")

Found links: {'links': [{'type': 'about page', 'url': 'https://tourno.fi/about_tournoco.php'}]}


# Welcome to Tourno.fi! 🌟⚽️

**Where Goals Are Scored and Friends Are Made**  
(And where occasionally, shirts are stained with grass!)

---

## What We Do

At **Tourno.fi**, we’re not just selling tournaments; we’re hawking the thrill of victory and the agony of minor scrapes! With **27 tournaments, 954 teams**, and a whopping **8265 goals** scored, we’re here to ensure that your weekends are filled with friendly competition—or in some cases, fierce rivalries. Remember: the grass may be greener on the other side, but it’s definitely a lot dirtier after a match!

---

## Fun & Games (Literally)

### Our Stats:
- **Number of Tournaments:** 27
- **Total Teams:** 954
- **Games Played:** 2064
- **Goals Made:** 8265  
(At least half of those were probably **accidentally** scored by enthusiastic parents yelling “GO, GO, GO!”)

### Upcoming Tournaments:
1. **KodinTavaratalo Cup 2025**  
   - **Date:** 4-5 Oct 2025  
   - Join us, and if you register before 31.3.2025, enjoy a 20% discount! (Meant for saving money, but likely to be spent on team snacks instead!)

2. **ViTa Syysturnaus**  
   - **Date:** 14-15 Sept 2024  
   - Perfect for forgetting those awkward childhood moments (like missing a penalty kick!)

---

## Customers We Love 💖

We’re like that one friend who always shows up with pizza. Our customers range from competitive clubs like **HJK and FC Haka** to enthusiastic little leagues where everyone gets a chance to shine (and whether they shine is debatable). 

Join the ranks of legends like:
- BK-46
- Espoon Tikka
- FC Honka  
(Plus many more! If we named them all, we might run out of space. Don't worry; they don’t mind!)

---

## Join Our Squad! 🏅

**Looking for a new gig?**  
At Tourno.fi, we value a culture that’s half laid-back and half adrenaline-pumping. Think you can handle that? 

We’re always on the lookout for:
- **Tournament Planners** (Must like spreadsheets, snacks, and wearing a whistle!)
- **Customer Service Ninjas** (Who can handle demands and complaints swiftly. Also, pizza delivery may happen.)
- **Marketing Wizards** (To make sure everyone knows how cool we are!)

If you want to work somewhere you can brainstorm great ideas **AND** join in on the occasional office match: this is the place for you!

---

## Why Tourno.fi? 🤔

You might ask, why join us?
- **We Manage The Fun**: You focus on winning, we do the rest!
- **Flexible Culture**: From serious tournaments to goofy team challenges, we want your experience to be memorable (even if that includes embarrassing dance moves).
- **Networking Opportunities**: Meet fellow enthusiasts and coaches! Who knows? Your next business partner might be on the opposing team!

---

**So what are you waiting for?**  
Gather your team, lace up those (maybe slightly mismatched) cleats, and join us at Tourno.fi! ⚽💼🎉 

---

**Join the Goal-Scoring Adventure!**  
For more info, visit [Tourno.fi](https://tourno.fi) - “Your Ultimate Destination for Tournaments!”  🏆

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>